# RL Self‑Play Lab — Notebook

## Train an agent to play Connect4 using self-play. Evaluate against a random agent. Play against rourself. Update the training based on playing with yourself.

## Install Dependencies

In [ ]:
!pip install pettingzoo supersuit gymnasium numpy

## Imports

In [ ]:
from pettingzoo.classic import connect_four_v3
import numpy as np
from collections import defaultdict

# Environment Exploration

In [ ]:
env = connect_four_v3.env()
env.reset()
print(env.agents)
for agent in env.agent_iter():
    obs, reward, termination, truncation, info = env.last()
    if termination or truncation:
        print('Over')
        env.step(None)
    else:
        action = env.action_space(agent).sample()
        env.step(action)
        print(agent, ' chooses ', action)



# Q‑table and Helper Functions

In [ ]:
Q = defaultdict(lambda: np.zeros(7))
def obs_to_state(obs):
    return tuple(obs['observation'].flatten())
gamma = 0.99
lr = 0.1
epsilon = 0.2

def select_action(state, obs):
    if np.random.random() < epsilon:
        return np.random.randint(7)
    return np.argmax(Q[state]) 

# Self‑Play Training Loop

In [ ]:
env = connect_four_v3.env()
wins = draws = losses = 0
for episode in range(2000):
    env.reset()
    
    last_state = {agent: None for agent in env.agents}
    last_action = {agent: None for agent in env.agents}

    for agent in env.agent_iter():
        obs, reward, termination, truncation, info = env.last()
        # TERMINAL TURN ----
        if termination or truncation:
            # Final Q-update for THIS agent (if it previously acted)
            if last_state[agent] is not None:
                Q[last_state[agent]][last_action[agent]] += **fill in **
            # Must pass None after termination
            env.step(None)

            # Keep track
            if reward == 1: wins += 1
            elif reward == 0: draws += 1
            else: losses += 1
            continue
        # NORMAL TURN ----
        state = obs_to_state(obs)
        action = select_action(state, obs)
        
        # Update *this* agent's previous step based on current reward
        if last_state[agent] is not None:
            **fill in **

        # Store for next round
        last_state[agent] = state
        last_action[agent] = action


        # Step environment
        env.step(action)


print(wins, draws, losses)

## Do you see an error saying Illegal Move? Update the select_action function above!!

# Evaluation Function (against RANDOM)

In [ ]:
def agent_policy(obs):
    state = tuple(obs["observation"].flatten())
    legal = ** fill in **
    qvals = ** fill in **
    return ** fill in **

def evaluate_once():
    env = connect_four_v3.env()
    env.reset()

    for agent in env.agent_iter():
        obs, reward, termination, truncation, info = env.last()

        if termination or truncation:            
            if reward == 1:
                winner = agent                # this agent won
            elif reward == -1:
                # the other agent won
                winner = [a for a in env.agents if a != agent][0]
            else:
                winner = "draw"

            env.step(None)
            continue

        legal = np.where(obs["action_mask"] == 1)[0]

        if agent == "player_0":
            action = agent_policy(obs)      # trained agent
        else:
            action = np.random.choice(legal) # random opponent

        env.step(action)
    print("Winner:", winner)
    return reward # reward for last agent

In [ ]:
evaluate_once() 

# Play against HUMAN

In [ ]:

def human_move(obs):
    legal = np.where(obs["action_mask"] == 1)[0]
    print("Legal moves:", legal.tolist())

    while True:
        move = input("Your move (0-6): ")

        # validate input
        if not move.isdigit():
            print("Please enter a number 0-6.")
            continue

        move = int(move)

        if move in legal:
            return move
        else:
            print("Illegal move! Choose from:", legal.tolist())


def play_vs_human():
    env = connect_four_v3.env(render_mode="human")
    env.reset()

    print("\nStarting Connect4: YOU = player_0, AGENT = player_1\n")

    for agent in env.agent_iter():
        obs, reward, termination, truncation, info = env.last()

        # Render board state
        env.render()

        if termination or truncation:
            print("\n--- GAME OVER ---")
            if reward == 1:
                if agent == "player_0":
                    print(f"YOU WIN!")
                else:
                    print(f"AGENT WIN!")
            elif reward == -1:
                other = [a for a in env.agents if a != agent][0]
                if other == "player_0":
                    print(f"YOU WIN!")
                else:
                    print(f"AGENT WIN!")
            else:
                print("DRAW!")
            env.step(None)
            break

        # Human turn
        if agent == "player_0":
            print("\nYour turn (player_0):")
            action = human_move(obs)

        # Agent turn
        else:
            print("\nAgent thinking...")
            action = agent_policy(obs)
            print(f"Agent chooses: {action}")

        env.step(action)


# Run the game
play_vs_human()


# Saving the agent

In [ ]:

import pickle
with open("connect4_agent.pkl", "wb") as f:
    pickle.dump(Q, f)


# Loading the agent

In [ ]:

with open("connect4_agent.pkl", "rb") as f:
    Q = pickle.load(f)


# Update the training with Human play!

In [ ]:

import copy
Q_before = copy.deepcopy(Q)

# --------------------------
# HUMAN TRAINING LOOP
# --------------------------
def play_and_train_vs_human():
    env = connect_four_v3.env(render_mode="human")
    env.reset()

    # Store last state/action per agent
    last_state = {agent: None for agent in env.agents}
    last_action = {agent: None for agent in env.agents}

    print("\nGAME STARTED – You are player_0!\n")

    for agent in env.agent_iter():
        obs, reward, termination, truncation, info = env.last()
        env.render()

        # --------------------------
        # TERMINAL STATE
        # --------------------------
        if termination or truncation:

            # FINAL Q-update for the agent who just played last time
            if last_state[agent] is not None:
                Q[last_state[agent]][last_action[agent]] += lr * (
                    reward - Q[last_state[agent]][last_action[agent]]
                )

            # Announce winner
            if reward == 1:
                if agent == "player_0":
                    print(f"YOU WIN!")
                else:
                    print(f"AGENT WIN!")
            elif reward == -1:
                other = [a for a in env.agents if a != agent][0]
                if other == "player_0":
                    print(f"YOU WIN!")
                else:
                    print(f"AGENT WIN!")
            else:
                print("\nDRAW!")

            env.step(None)
            break

        # --------------------------
        # NON-TERMINAL STATE
        # --------------------------
        state = obs_to_state(obs)

        # Choose action
        if agent == "player_0":
            print("\nYour turn:")
            action = human_move(obs)
        else:
            print("\nAgent turn:")
            action = agent_policy(obs)
            print("Agent chooses:", action)

        # --------------------------
        # Q-UPDATE FROM PREVIOUS TURN
        # --------------------------
        **fill in **

        # store state-action for next update
        last_state[agent] = state
        last_action[agent] = action

        # Play the action
        env.step(action)

    print("\nQ-table updated from this game!\n")


# Run it
play_and_train_vs_human()


# Visualize changes in Q-table

In [ ]:

def count_changed(Q_before, Q_after, tol=1e-9):
    changed = 0
    for state in Q_after:
        if state not in Q_before:
            changed += len(Q_after[state])  # newly created rows
        else:
            changed += np.sum(np.abs(Q_after[state] - Q_before[state]) > tol)
    return changed

changed_count = count_changed(Q_before, Q)
print("Number of Q-values that changed:", changed_count)


def q_difference_norm(Q_before, Q_after):
    total = 0.0
    for state in Q_after:
        if state not in Q_before:
            total += np.sum(np.abs(Q_after[state]))
        else:
            total += np.sum(np.abs(Q_after[state] - Q_before[state]))
    return total

diff = q_difference_norm(Q_before, Q)
print("Total L1 difference:", diff)

def print_q_changes(Q_before, Q_after, max_states=20):
    print("=== Q-table changes ===")
    count = 0
    for state in Q_after:
        if state not in Q_before:
            print("\nNew state added:", state)
            print("Values:", Q_after[state])
            continue
        
        delta = Q_after[state] - Q_before[state]
        if np.any(np.abs(delta) > 1e-9):
            print("\nState:", state)
            print("Before:", Q_before[state])
            print("After :", Q_after[state])
            print("Delta :", delta)
            count += 1
            if count >= max_states:
                break
print_q_changes(Q_before, Q)

In [ ]:
import matplotlib.pyplot as plt

def plot_q_change(Q_before, Q_after):
    diffs = []
    for state in Q_after:
        if state not in Q_before:
            diffs.extend(np.abs(Q_after[state]))
        else:
            diffs.extend(np.abs(Q_after[state] - Q_before[state]))

    plt.hist(diffs, bins=20)
    plt.title("Distribution of Q-value changes")
    plt.xlabel("|dQ|")
    plt.ylabel("Count")
    plt.show()